In [3]:
import pandas as pd

In [4]:
df=pd.read_csv('Reviews1.csv')
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8187 entries, 0 to 8186
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Id                      8187 non-null   int64 
 1   ProductId               8187 non-null   object
 2   UserId                  8187 non-null   object
 3   ProfileName             8187 non-null   object
 4   HelpfulnessNumerator    8187 non-null   int64 
 5   HelpfulnessDenominator  8187 non-null   int64 
 6   Score                   8187 non-null   int64 
 7   Time                    8187 non-null   int64 
 8   Summary                 8187 non-null   object
 9   Text                    8187 non-null   object
dtypes: int64(5), object(5)
memory usage: 639.7+ KB


In [6]:
df.isna().sum()


,0
Id,0
ProductId,0
UserId,0
ProfileName,0
HelpfulnessNumerator,0
HelpfulnessDenominator,0
Score,0
Time,0
Summary,0
Text,0


In [7]:


# Select only the columns we care about
df = df[['Text', 'Score']].dropna()

# Convert score to sentiment label
def score_to_label(score):
    if score <= 2:
        return 0  # Negative
    elif score == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

df['label'] = df['Score'].apply(score_to_label)

# Shuffle the data
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Show sample
print(df[['Text', 'label']].head())


                                                Text  label
0  I am a mom of 3 kids. With my older two I had ...      2
1  After my husband's enthusiastic reaction to th...      2
2  I heard my kids talking about candy they remem...      2
3  If you like Orange Crush you may become a fan,...      0
4  As another reviewer stated, many of the review...      0


In [8]:
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split
import torch

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenization function
def tokenize_data(texts, labels, max_len=128):
    encoding = tokenizer.batch_encode_plus(
        texts.tolist(),
        add_special_tokens=True,
        padding='max_length',
        max_length=max_len,
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )
    return encoding['input_ids'], encoding['attention_mask'], torch.tensor(labels.tolist())

# Prepare inputs
texts = df['Text']
labels = df['label']

# Split data
X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=0.1, random_state=42)

# Tokenize
train_input_ids, train_attention_masks, train_labels = tokenize_data(X_train, y_train)
val_input_ids, val_attention_masks, val_labels = tokenize_data(X_val, y_val)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [9]:
from torch.utils.data import Dataset, DataLoader

# Define custom dataset
class BERTDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'labels': self.labels[idx]
        }

# Create datasets
train_dataset = BERTDataset(train_input_ids, train_attention_masks, train_labels)
val_dataset = BERTDataset(val_input_ids, val_attention_masks, val_labels)

# Create dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=16)


In [10]:
from transformers import BertForSequenceClassification, get_scheduler
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from tqdm import tqdm
# Load pre-trained BERT with classification head
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)
model = model.to(device)

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs =1
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

# Loss function
loss_fn = CrossEntropyLoss()

# Training loop
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    model.train()
    total_loss = 0

    for batch in tqdm(train_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            labels=batch['labels']
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    avg_loss = total_loss / len(train_dataloader)
    print(f"Training loss: {avg_loss:.4f}")


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/1


100%|██████████| 461/461 [02:26<00:00,  3.16it/s]

Training loss: 0.4516


In [11]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import numpy as np

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(val_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask']
        )

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['labels'].cpu().numpy())

# النتائج
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
cm = confusion_matrix(all_labels, all_preds)

print(f"\n✅ Accuracy: {accuracy:.4f}")
print(f"🎯 F1 Score: {f1:.4f}")
print("\n📊 Confusion Matrix:\n", cm)
print("\n📄 Classification Report:\n", classification_report(all_labels, all_preds, target_names=["Negative", "Neutral", "Positive"]))


100%|██████████| 52/52 [00:05<00:00,  9.53it/s]


✅ Accuracy: 0.8779
🎯 F1 Score: 0.8716

📊 Confusion Matrix:
 [[100   5  15]
 [ 32  13  12]
 [ 28   8 606]]

📄 Classification Report:
               precision    recall  f1-score   support

    Negative       0.62      0.83      0.71       120
     Neutral       0.50      0.23      0.31        57
    Positive       0.96      0.94      0.95       642

    accuracy                           0.88       819
   macro avg       0.69      0.67      0.66       819
weighted avg       0.88      0.88      0.87       819



In [22]:
from transformers import BertTokenizer

# Save model
model_save_path = "bert_sentiment_model"
model.save_pretrained(model_save_path)

# Save tokenizer as well
tokenizer.save_pretrained(model_save_path)

print(f"✅ Model and tokenizer saved to: {model_save_path}")


✅ Model and tokenizer saved to: bert_sentiment_model


In [23]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

# تحميل النموذج والتوكنيزر
model_path = "bert_sentiment_model"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model = model.to(device)
model.eval()

# دالة التنبؤ
def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        pred_class = torch.argmax(probs, dim=1).item()

    label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}
    return label_map[pred_class], probs[0][pred_class].item()

# تجربة على نص جديد
text = "Product arrived labeled as Jumbo Salted Peanut.."
label, confidence = predict_sentiment(text)
print(f"🔍 Prediction: {label} ({confidence*100:.2f}%)")


🔍 Prediction: Negative (59.22%)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')